# LSSTCam: AOS tools

Owner: **Chris  Suberlak** [@suberlak](https://github.com/lsst-ts/ts_aos_analysis/issues/new?body=@suberlak) <br>
Last Verified to Run: **2025-04-22** <br>
Software Versions:
  - `lsst_distrib`: **w_2025_15**
  - `ts_wep`:**v14.2.0**

We have `raw` images simulated with `imSim` in `aos_imsim` repo. They consist of  a full-array (197 detectors) focus triplet, i.e. two defocal exposures (seqNum 960,962), and one in-focus exposure (seqNum961).  

In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt 
import numpy as np 
import os 
from lsst.daf.butler import Butler
from lsst.obs.lsst import LsstCam
from lsst.ts.wep.utils import runProgram
from lsst.ts.wep.task.generateDonutDirectDetectTask import (
GenerateDonutDirectDetectTask,GenerateDonutDirectDetectTaskConfig)
from lsst.ts.wep.task.cutOutDonutsCwfsTask import (CutOutDonutsCwfsTask, CutOutDonutsCwfsTaskConfig)
from lsst.ts.wep.task.cutOutDonutsScienceSensorTask import (CutOutDonutsScienceSensorTask, CutOutDonutsScienceSensorTaskConfig)
from lsst.ts.wep.task.reassignCwfsCutoutsTask import (ReassignCwfsCutoutsTask, ReassignCwfsCutoutsTaskConfig)
from lsst.ts.wep.task.fitDonutRadiusTask import (FitDonutRadiusTask, FitDonutRadiusTaskConfig)

In [ ]:
butlerRootPath = '/sdf/group/rubin/repo/aos_imsim'
butler = Butler(butlerRootPath)
dataRefs = list(butler.registry.queryDatasets('raw', collections=['LSSTCam/raw/all'],
                where="instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num = 960").expanded())
len(dataRefs)

In [ ]:
dataRefs = list(butler.registry.queryDatasets('raw', collections=['LSSTCam/raw/all'],
                where="instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num = 961").expanded())
len(dataRefs)


In [ ]:
dataRefs = list(butler.registry.queryDatasets('raw', collections=['LSSTCam/raw/all'],
                where="instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num = 962").expanded())
len(dataRefs)


# Run ISR for entire triplet 
Run ISR with variance calculation:


    instrument: lsst.obs.lsst.LsstCam
    tasks:
      isr:
        class: lsst.ip.isr.isrTask.IsrTask
        config:
          connections.outputExposure: 'postISRCCD'
          doBias: False
          doVariance: True
          doLinearize: False
          doCrosstalk: False
          doDefect: False
          doNanMasking: False
          doInterpolate: False
          doBrighterFatter: False
          doDark: False
          doFlat: False
          doApplyGains: True
          doFringe: False
          doOverscan: True
          python: OverscanCorrectionTask.ConfigClass.fitType = 'MEDIAN'


Saved in `/sdf/group/rubin/shared/scichris/DM-41957_lsstCam_sweep/lsstPipelineISRvar.yaml` 

In [ ]:
numPro=5
pipelineYamlPath = '/sdf/group/rubin/shared/scichris/DM-41957_lsstCam_sweep/lsstPipelineISRvar.yaml'
day_obs = 20280818
isrRunName = f'u/scichris/runIsr_lsstCam_{day_obs}_triplet'
print('\n')

butlerRootPath = '/sdf/group/rubin/repo/aos_imsim'
butlerInstName = 'Cam'
expression = f"exposure.day_obs={day_obs} "
cmd = f"pipetask run -b {butlerRootPath} "\
      f"-i LSST{butlerInstName}/raw/all,LSST{butlerInstName}/calib/unbounded "\
      f"--instrument lsst.obs.lsst.Lsst{butlerInstName} "\
      f"--register-dataset-types --output-run {isrRunName}  -p {pipelineYamlPath} -d "\
      f'"{expression}"'\
      f" -j {numPro} "
print(cmd)



This can be run inline if needed: 

In [ ]:
runProgram(cmd)

# Run donut detection , cutouts,  radius task in the notebook

Steps for in-focus exposure using [CWFS donuts](https://github.com/lsst-ts/donut_viz/blob/develop/pipelines/production/lsstCamRapidAnalysisPipeline_Danish.yaml)


  - generateDonutDirectDetectTask
  - cutOutDonutsCwfsTask
  - reassignCwfsCutoutsTask
  - fitDonutRadiusTask
  - calcZernikesTask


## A) Use CWFS donuts,   get PSF plot for science sensors 

Elements of [yaml](https://github.com/lsst-ts/donut_viz/blob/develop/pipelines/_ingredients/wepDirectDetectCwfsPipeline.yaml) reproduced in-notebook:

    description: wep direct detect pipeline for wavefront sensors
    tasks:
       generateDonutDirectDetectTask:
        class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
        config:
          donutSelector.useCustomMagLimit: True
          donutSelector.sourceLimit: 20
       cutOutDonutsCwfsTask:
        class: lsst.ts.wep.task.cutOutDonutsCwfsTask.CutOutDonutsCwfsTask
        config:
          # And here we specify the configuration settings originally defined in
          # CutOutDonutsCwfsTaskConfig.
          # Test CWFS pipeline works when specifying instrument parameters.
          donutStampSize: 160
          initialCutoutPadding: 40
       reassignCwfsCutoutsTask:
        class: lsst.ts.wep.task.reassignCwfsCutoutsTask.ReassignCwfsCutoutsTask

and [this yaml](https://github.com/lsst-ts/donut_viz/blob/develop/pipelines/_ingredients/donutVizGroupPipeline.yaml)

    description: donut viz pipeline tasks
    tasks:
      aggregateZernikeTablesTask:
        class: lsst.donut.viz.AggregateZernikeTablesTask
      aggregateDonutTablesCwfsTask:
        class: lsst.donut.viz.AggregateDonutTablesCwfsTask
      aggregateDonutStampsTask:
        class: lsst.donut.viz.AggregateDonutStampsTask
      aggregateAOSVisitTableCwfsTask:
        class: lsst.donut.viz.AggregateAOSVisitTableCwfsTask
      plotAOSTask:
        class: lsst.donut.viz.PlotAOSTask
        config:
          doRubinTVUpload: False
      plotDonutCwfsTask:
        class: lsst.donut.viz.PlotDonutCwfsTask
        config:
          doRubinTVUpload: False
      plotPsfZernTask:
        class: lsst.donut.viz.PlotPsfZernTask
        config:
          doRubinTVUpload: False

both imported [here](https://github.com/lsst-ts/donut_viz/blob/develop/pipelines/production/lsstCamRapidAnalysisPipeline_Danish.yaml) : 

    description: rapid analysis pipeline for LSSTCam
    instrument: lsst.obs.lsst.LsstCam
    imports:
      - $DONUT_VIZ_DIR/pipelines/_ingredients/wepDirectDetectCwfsPipeline.yaml
      - $DONUT_VIZ_DIR/pipelines/_ingredients/donutVizCwfsPipeline.yaml
    tasks:
      calcZernikesTask:
        class: lsst.ts.wep.task.calcZernikesTask.CalcZernikesTask
        config:
          python: |
            from lsst.ts.wep.task import EstimateZernikesDanishTask
            config.estimateZernikes.retarget(EstimateZernikesDanishTask)
          donutStampSelector.maxSelect: 5
          donutStampSelector.maxFracBadPixels: 2.0e-4
          estimateZernikes.binning: 4
          estimateZernikes.nollIndices:
            [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28]
          estimateZernikes.saveHistory: False
          estimateZernikes.lstsqKwargs:
            ftol: 1.0e-3
            xtol: 1.0e-3
            gtol: 1.0e-3
    
    # Define pipeline steps
    subsets:
      step1:
        subset:
          - isr
          - generateDonutDirectDetectTask
          - cutOutDonutsCwfsTask
          - reassignCwfsCutoutsTask
          - calcZernikesTask
        description: |
          This step processes the input images with ISR,
          finds and cuts out the donut stamps,
          and estimates the Zernike coefficients from the donut pairs.
      step2a:
        subset:
          - aggregateZernikeTablesTask
          - aggregateDonutTablesCwfsTask
          - aggregateAOSVisitTableCwfsTask
          - plotAOSTask
          - aggregateDonutStampsTask
          - plotDonutCwfsTask
        description: |
          AOS Donut visualization plotting tasks. This step generates plots
          (including the pyramid residual and donut gallery) and
          tables for the AOS visit.

In [ ]:
butler = Butler('/sdf/group/rubin/repo/aos_imsim', 
                collections=['u/scichris/runIsr_lsstCam_20280818_triplet'])
dataRefs = butler.query_datasets('postISRCCD' ,collections=['u/scichris/runIsr_lsstCam_20280818_triplet'],
                        where="instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num = 961\
                        and detector.purpose = 'WAVEFRONT' and detector.id in (191,192)"
                                )
len(dataRefs)

In [ ]:
dataRefs

In [ ]:
camera = LsstCam.getCamera()

In [ ]:
exposure_intra = butler.get('postISRCCD', dataId = dataRefs[0].dataId,collections=['u/scichris/runIsr_lsstCam_20280818_triplet'])
exposure_extra = butler.get('postISRCCD', dataId = dataRefs[1].dataId,collections=['u/scichris/runIsr_lsstCam_20280818_triplet'])

In [ ]:
config = GenerateDonutDirectDetectTaskConfig()
config.donutSelector.useCustomMagLimit = True
task= GenerateDonutDirectDetectTask(config=config)
taskOutExtra =  task.run(exposure_extra,camera)

In [ ]:
config = GenerateDonutDirectDetectTaskConfig()
config.donutSelector.useCustomMagLimit = True
task= GenerateDonutDirectDetectTask(config=config)
taskOutIntra =  task.run(exposure_intra,camera)

In [ ]:
config = CutOutDonutsCwfsTaskConfig() 
task = CutOutDonutsCwfsTask(config=config)

taskCutOutIntra = task.run(exposure_intra, taskOutIntra.donutCatalog, camera) 
taskCutOutExtra = task.run(exposure_extra, taskOutExtra.donutCatalog, camera)

In this example running the task in-notebook it doesn't seem necessary to run the `reassign` task, since we have direct access to the stamps: 

In [ ]:
config = FitDonutRadiusTaskConfig()
task = FitDonutRadiusTask(config=config)
taskOutFit = task.run(taskCutOutExtra.donutStampsExtra, taskCutOutIntra.donutStampsIntra)

In [ ]:
taskOutFit.donutRadiiTable

Illustrate the fitted radius by plotting the available stamps:

In [ ]:
donutRadiiTable = taskOutFit.donutRadiiTable

In [ ]:

def plot_stamps_radii(stamps, donutRadiiTable, nrows=3, ncols=4,
                     defocal ='extra'):


    m = donutRadiiTable['DFC_TYPE'] == 'extra'
    visit = donutRadiiTable[m]['VISIT'][0]
    
    ncells = nrows * ncols 
    if ncells < len(stamps):
        print('Warning: insufficient number of cells to plot all stamps')
        print(f'Using a subset of {ncells} / {len(stamps)}')
              
    fig,axs = plt.subplots(nrows, ncols, figsize=(2*ncols,2*nrows))
    ax = np.ravel(axs)
    
    i=0
    for stamp in stamps:
        image = stamp.stamp_im.image.array
        ax[i].imshow(image, origin='lower')
        
        x_left_edge = donutRadiiTable[m][i]['X_LEFT_EDGE']
        x_right_edge = donutRadiiTable[m][i]['X_RIGHT_EDGE']
        
        for x_edge in [x_left_edge, x_right_edge]:
            ax[i].axvline(x_edge, ymin=0.25, ymax=0.75, ls='--', c='white', lw=2)
        
        ax[i].text(100,100,i,fontsize=14, color='white')
        donut_radius= donutRadiiTable[m][i]['RADIUS']
        ax[i].text(50,10,f'r={donut_radius:.2f}',fontsize=14, color='white')      
    
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        i+=1


    fig.subplots_adjust(hspace=0.05, wspace=0.05)
    
    if len(stamps)<len(ax):
        for i in range(len(stamps), len(ax)):
            ax[i].axis('off')
    ax1 = fig.add_axes([1,0.2,0.4,0.6])
        
    radii = donutRadiiTable[m]['RADIUS']
    ax1.vlines(radii,  colors='blue',ymin=0, ymax=1, linestyles='solid', label="In final_select")
    ax1.set_xlabel('donut radius [px]')
    ax1.set_yticks([])
    fig.suptitle(f'{visit}, cutoff at {config.widthMultiplier} FWHM '+r'$\sigma$')
    

stamps = taskCutOutExtra.donutStampsExtra
plot_stamps_radii(stamps, donutRadiiTable, nrows=2, ncols=4,
                     defocal ='extra')

Run as a pipetask:

    description: wep direct detect and fit donut radius
    instrument: lsst.obs.lsst.LsstCam
    tasks:
      generateDonutDirectDetectTask:
        class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
        config:
          donutSelector.useCustomMagLimit: True
          donutSelector.sourceLimit: 20
      cutOutDonutsCwfsTask:
        class: lsst.ts.wep.task.cutOutDonutsCwfsTask.CutOutDonutsCwfsTask
        config:
          # And here we specify the configuration settings originally defined in
          # CutOutDonutsCwfsTaskConfig.
          # Test CWFS pipeline works when specifying instrument parameters.
          donutStampSize: 160
          initialCutoutPadding: 40
      reassignCwfsCutoutsTask:
        class: lsst.ts.wep.task.reassignCwfsCutoutsTask.ReassignCwfsCutoutsTask
      fitDonutRadiusTask:
        class: lsst.ts.wep.task.fitDonutRadiusTask.FitDonutRadiusTask
      
     

Test on  a single detector:

    pipetask run -b /sdf/group/rubin/repo/aos_imsim -i LSSTCam/calib/unbounded,u/scichris/runIsr_lsstCam_20280818_triplet --instrument lsst.obs.lsst.LsstCam --register-dataset-types --output-run u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_det191-192_test  -p /sdf/group/rubin/shared/scichris/lsstCam_first_light/lsstPipelineFitRadiusCwfs.yaml -d "instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num = 961 and detector.purpose = 'WAVEFRONT' and detector.id in (191,192)" -j 5 

Test the output:

In [ ]:
output_collection = 'u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_det191-192_test'
butler  = Butler('/sdf/group/rubin/repo/aos_imsim',
                 collections=[output_collection]
                )
dataRefs = butler.query_datasets('donutRadiiTable', collections=[output_collection])
donutRadiiTable = butler.get('donutRadiiTable', dataId=dataRefs[0].dataId,
                             collections=[output_collection]
                            )

In [ ]:
donutRadiiTable

Run for all wavefront detectors:

    pipetask run -b /sdf/group/rubin/repo/aos_imsim -i LSSTCam/calib/unbounded,u/scichris/runIsr_lsstCam_20280818_triplet --instrument lsst.obs.lsst.LsstCam --register-dataset-types --output-run u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_detCwfs_test  -p /sdf/group/rubin/shared/scichris/lsstCam_first_light/lsstPipelineFitRadiusCwfs.yaml -d "instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num = 961 and detector.purpose = 'WAVEFRONT' " -j 5

## B) Use science sensor donuts (FAM) mode 


Steps for FAM defocal pair using science sensors for donuts 

  - generateDonutDirectDetectTask
  - cutOutDonutsScienceSensorTask
  - fitDonutRadiusTask
  - calcZernikesTask

In [ ]:
config = GenerateDonutDirectDetectTaskConfig()
task= GenerateDonutDirectDetectTask(config=config)

In [ ]:
butler = Butler('/sdf/group/rubin/repo/aos_imsim', collections=['u/scichris/runIsr_lsstCam_20280818_triplet'])

In [ ]:
dataRefs = butler.query_datasets('postISRCCD' ,collections=['u/scichris/runIsr_lsstCam_20280818_triplet'],
                        where="instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num in (960,962)\
                        and detector.purpose = 'SCIENCE' and detector.id=98"
                                )

In [ ]:
len(dataRefs)

In [ ]:
dataRefs

In [ ]:
camera = LsstCam.getCamera()

In [ ]:
exposure_intra = butler.get('postISRCCD', dataId = dataRefs[0].dataId,collections=['u/scichris/runIsr_lsstCam_20280818_triplet'])
exposure_extra = butler.get('postISRCCD', dataId = dataRefs[1].dataId,collections=['u/scichris/runIsr_lsstCam_20280818_triplet'])

In [ ]:
taskOutExtra =  task.run(exposure_extra,camera)

In [ ]:
taskOutIntra = task.run(exposure_intra,camera)

In [ ]:
config = CutOutDonutsScienceSensorTaskConfig() 
task = CutOutDonutsScienceSensorTask(config=config)

taskCutOut = task.run([exposure_intra,exposure_extra],
                   [taskOutIntra.donutCatalog, taskOutExtra.donutCatalog],
                   camera)

Use these stamps to run `fitDonutRadius` task: 

In [ ]:
config = FitDonutRadiusTaskConfig()
task = FitDonutRadiusTask(config=config)
taskOutFit = task.run(taskCutOut.donutStampsExtra, taskCutOut.donutStampsIntra)

In [ ]:
taskOutFit.donutRadiiTable

Plot the stamps and fitted radii:

In [ ]:
stamps = taskCutOut.donutStampsExtra
donutRadiiTable =  taskOutFit.donutRadiiTable
plot_stamps_radii(stamps, donutRadiiTable, nrows=3, ncols=4,
                 defocal ='extra')

This task can be run before any sort of `donutQualityTable` is available from `calcZernikesTask`. But all the selection components are available from `donutStamps.metadata`.

Run all of these as a pipetask: 


    description: wep direct detect and fit donut radius
    instrument: lsst.obs.lsst.LsstCam
    tasks:
     generateDonutDirectDetectTask:
        class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
        config:
          donutSelector.useCustomMagLimit: True
          donutSelector.sourceLimit: 20
      cutOutDonutsScienceSensorGroupTask:
        class: lsst.ts.wep.task.cutOutDonutsScienceSensorTask.CutOutDonutsScienceSensorTask
        config:
          python: |
            from lsst.ts.wep.task.pairTask import GroupPairer
            config.pairer.retarget(GroupPairer)
          donutStampSize: 200
          initialCutoutPadding: 40
    
      fitDonutRadiusTask:
        class: lsst.ts.wep.task.fitDonutRadiusTask.FitDonutRadiusTask
    

Test on a single detector:

    pipetask run -b /sdf/group/rubin/repo/aos_imsim -i LSSTCam/calib/unbounded,u/scichris/runIsr_lsstCam_20280818_triplet --instrument lsst.obs.lsst.LsstCam --register-dataset-types --output-run u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_det98_test  -p /sdf/group/rubin/shared/scichris/lsstCam_first_light/lsstPipelineFitRadius.yaml -d "instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num in (960,962) and detector.purpose = 'SCIENCE' and detector.id=98" -j 5 



Inspect outputs:

In [ ]:
output_collection = 'u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_det98_test'
butler  = Butler('/sdf/group/rubin/repo/aos_imsim',
                 collections=[output_collection]
                )
dataRefs = butler.query_datasets('donutRadiiTable', collections=[output_collection])
donutRadiiTable = butler.get('donutRadiiTable', dataId=dataRefs[0].dataId,
                             collections=[output_collection]
                            )

In [ ]:
donutRadiiTable

Given that this worked fine, submit more detectors: 

    pipetask run -b /sdf/group/rubin/repo/aos_imsim -i LSSTCam/calib/unbounded,u/scichris/runIsr_lsstCam_20280818_triplet --instrument lsst.obs.lsst.LsstCam --register-dataset-types --output-run u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_960_962_science_test  -p /sdf/group/rubin/shared/scichris/lsstCam_first_light/lsstPipelineFitRadius.yaml -d "instrument='LSSTCam' and exposure.day_obs=20280818 and exposure.seq_num in (960,962) and detector.purpose = 'SCIENCE' " -j 5 

Show summary for multiple detectors: 

In [ ]:
dataRefs = butler.query_datasets('donutRadiiTable', 
        collections=['u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_960_962_science_test']
                                )

In [ ]:
# read in all tables as a dictionary
tables = {}
for ref in dataRefs:
    tables[ref.dataId['detector']] = butler.get('donutRadiiTable', 
                                                dataId = ref.dataId,
        collections=['u/scichris/runIsr_lsstCam_20280818_triplet_donutStampsRadii_960_962_science_test']
                                               )

In [ ]:
# calculate median radius per defocal type per detector 
detIds = []
dfcTypes = []
medianRadii = []
for detId in tables.keys():
    table = tables[detId]
    # only select donuts where fit succeeded 
    m = table['FAIL_FLAG'] == 0
    
    # Group the table by 'DFC_TYPE'
    grouped = table[m][['DFC_TYPE','RADIUS']].group_by('DFC_TYPE')
    
    # Calculate the median 'RADIUS' per group
    median_table = grouped.groups.aggregate(np.median)
    if  len(median_table) == 2 :
        #print(detId, median_table)
        detIds.append([detId,detId])
        dfcTypes.append(median_table['DFC_TYPE'].data)
        medianRadii.append(median_table['RADIUS'].data)

detIdsFlat = np.concatenate([np.asarray(item) for item in detIds])
dfcTypesFlat = np.concatenate([np.asarray(item) for item in dfcTypes])
medianRadiiFlat =  np.concatenate([np.asarray(item) for item in medianRadii])

results = Table(data=[detIdsFlat, dfcTypesFlat, medianRadiiFlat],
      names = ['DETID', 'DFC_TYPE','MEDIAN_RADIUS',]
     )

Plot histogram of results:

In [ ]:
for dfc in ['intra', 'extra']:
    m = results['DFC_TYPE'] == dfc
    plt.hist(results[m]['MEDIAN_RADIUS'], label=dfc, histtype='step', bins=35)
plt.xlabel('MEDIAN RADIUS')
plt.ylabel('COUNT')